In [16]:
import os, json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
load_dotenv()

True

In [2]:
model = ChatOpenAI(
    model="gpt-4o-mini"
)


In [8]:
response =model.invoke('what is my emi for this loan id BFL2024001"')

In [10]:
response.response_metadata

{'token_usage': {'completion_tokens': 149,
  'prompt_tokens': 21,
  'total_tokens': 170,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0,
   'cache_write_tokens': None,
   'cached_tokens': 0}},
 'model_provider': 'openai',
 'model_name': 'gpt-4o-mini-2024-07-18',
 'system_fingerprint': 'fp_3817d51ca9',
 'id': 'chatcmpl-E9wC7BXxwTywnGNVq0hbgVAbsq2bV',
 'service_tier': 'default',
 'finish_reason': 'stop',
 'logprobs': None}

In [13]:
with open("bajaj_db.json") as f:
    db = json.load(f)

In [18]:
# response
@tool
def get_loan_status(loan_id: str) -> dict:
    """Fetches current status of a Bajaj Finance loan from the database.
    Returns EMI amount, remaining months, outstanding balance, next due date.
    Use this when customer asks about their loan details, EMI, or balance.
    Args:
        loan_id: The loan account number (e.g., 'BFL2024001')
    """
    if loan_id not in db["loans"]:
        return {"error": f"Loan {loan_id} not found"}
    loan = db["loans"][loan_id]
    return {
        "customer_name": loan["customer_name"],
        "emi": loan["emi"],
        "remaining_months": loan["remaining_months"],
        "outstanding": loan["outstanding"],
        "next_due_date": loan["next_due_date"],
        "prepayment_charge_pct": loan["prepayment_charge_pct"]
    }

get_loan_status.invoke("BFL2024001")


{'customer_name': 'Rahul Tiwari',
 'emi': 8450,
 'remaining_months': 22,
 'outstanding': 185900,
 'next_due_date': '2026-05-05',
 'prepayment_charge_pct': 2.0}

In [19]:
tools = [get_loan_status]

In [ ]:
llm_with_tools = model.bind_tools(tools)


In [25]:
result = llm_with_tools.invoke('what is the status of loan id BFL9988')

In [28]:
result.tool_calls

[{'name': 'get_loan_status',
  'args': {'loan_id': 'BFL9988'},
  'id': 'call_TrwpLSZLacvjqQgcYvKpgggE',
  'type': 'tool_call'}]

In [30]:
tool_map = {
    "get_loan_status": get_loan_status
}

for tc in result.tool_calls:
    tool_name = tc["name"]
    tool_args = tc["args"]
    tool_map[tool_name].invoke(tool_args)
    print("result:", tool_map[tool_name].invoke(tool_args))

result: {'customer_name': 'Priya Mehta', 'emi': 24500, 'remaining_months': 204, 'outstanding': 2180000, 'next_due_date': '2026-05-10', 'prepayment_charge_pct': 0.0}
